# Depth Anything 3 — C++ Inference on Colab (GPU gate, Phase C)

Runs this repo's `inference_app` (ONNX Runtime backend) on a Colab GPU runtime:
build → export DA3-SMALL mono ONNX → `--task mono` → any-view pose batch (`--dynamo`) → `--task pose`.

Authored 2026-10-13, **PENDING-USER-RUN**: paste back the outputs of the last two cells as execution evidence.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!cmake --version | head -n 1
!python3 -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq cmake libopencv-dev ninja-build > /dev/null 2>&1
!pip install -q xformers huggingface_hub onnx onnxruntime pycolmap
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3 /content/Depth-Anything-3
!pip install -q -e /content/Depth-Anything-3 --no-deps

In [ ]:
!git clone https://github.com/olibartfast/depth-anything-3-cpp-inference /content/da3cpp  # post-merge master; pre-merge use -b <phase-branch>
!cmake -S /content/da3cpp -B /content/da3cpp/build -G Ninja -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build /content/da3cpp/build --parallel
!ctest --test-dir /content/da3cpp/build -R UnitTests --output-on-failure

In [ ]:
!mkdir -p /content/views && python3 - <<'EOF'
import cv2
from huggingface_hub import snapshot_download
d = snapshot_download('depth-anything/DA3-SMALL', local_dir='/content/da3small')
print('weights:', d)
EOF
!ls -la /content/da3small/

In [ ]:
!cd /content/da3cpp && python3 export_onnx.py --model-dir /content/da3small --task mono --process-res 280 --fixed-size --output /content/small-mono-280.onnx
!cd /content/da3cpp && python3 export_onnx.py --model-dir /content/da3small --task pose --dynamo --process-res 280 --output /content/small-pose-280.onnx

In [ ]:
!python3 - <<'EOF'
import cv2, glob
ims = sorted(glob.glob('/content/Depth-Anything-3/assets/examples/SOH/*.png'))[:2]
print('source views:', ims)
import shutil
for i, f in enumerate(ims):
    im = cv2.imread(f)
    h, w = im.shape[:2]; s = min(h, w)
    sq = cv2.resize(im[(h - s)//2:(h - s)//2 + s, (w - s)//2:(w - s)//2 + s], (280, 280))
    cv2.imwrite(f'/content/views/{i:02d}.jpg', sq)
print('wrote /content/views')
EOF
!ls /content/views/

In [ ]:
!/content/da3cpp/build/inference_app /content/small-mono-280.onnx /content/views/00.jpg --task mono --process-res 280 --output-dir /content/out-mono
!/content/da3cpp/build/inference_app /content/small-pose-280.onnx /content/views --task pose --process-res 280 --output-dir /content/out-pose
!cat /content/out-pose/poses.txt && echo --- && cat /content/out-pose/intrinsics.txt

## Evidence to paste back (user run)

- `nvidia-smi` GPU name, `ctest` result line, both export logs (output names), both inference logs, `poses.txt`/`intrinsics.txt` contents.
- TensorRT path is best-effort only (needs the pinned TRT 11.2.1.2 + CUDA 13.3 toolchain, not stock Colab).